# 🎓 Agent de Correction QCM — LangGraph + Groq




---
## Cellule 1 — Installation des dépendances

On installe les trois librairies nécessaires :
- **`langgraph`** : orchestration des agents sous forme de graphe d'état
- **`langchain-groq`** : connecteur LangChain pour l'API Groq
- **`langchain-core`** : prompts, messages, interface LLM abstraite
- **`python-dotenv`** : chargement de la clé API depuis un fichier `.env`

In [1]:
# Installation — à exécuter une seule fois
%pip install -q langgraph langchain-groq langchain-core python-dotenv

Note: you may need to restart the kernel to use updated packages.


---
## Cellule 2 — Configuration centrale

Tous les paramètres modifiables sont ici :
- **`GROQ_API_KEY`** : obtenir une clé gratuite sur https://console.groq.com
- **`MODEL_NAME`** : modèle Groq utilisé (`llama-3.3-70b-versatile`)
- **`TEMPERATURE`** : 0.1 → réponses stables et reproductibles
- **`ALPHA_CORRECT / ALPHA_INCORRECT`** : coefficients de mise à jour du niveau étudiant

> 💡 Créez un fichier `.env` à côté de ce notebook avec `GROQ_API_KEY=gsk_...`  
> ou renseignez la clé directement dans la variable ci-dessous.

In [2]:
import os
from dotenv import load_dotenv

# Charge la clé depuis .env si disponible
load_dotenv()

# ── Clé API Groq ──────────────────────────────────────────────────────────
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "VOTRE_CLE_GROQ_ICI")

# ── Paramètres du modèle ──────────────────────────────────────────────────
MODEL_NAME  = "llama-3.3-70b-versatile"   # Modèle Groq rapide et performant
TEMPERATURE = 0.1                          # Bas → correction déterministe
MAX_TOKENS  = 1000

# ── Paramètres de mise à jour du niveau étudiant ─────────────────────────
ALPHA_CORRECT   = 0.25   # Progression si bonne réponse
ALPHA_INCORRECT = 0.10   # Régression douce si mauvaise réponse
LEVEL_MIN       = 0.05   # Plancher du niveau
LEVEL_MAX       = 0.95   # Plafond du niveau

# ── Initialisation du LLM Groq ────────────────────────────────────────────
from langchain_groq import ChatGroq

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    api_key=GROQ_API_KEY
)

print(f"✅ LLM initialisé : {MODEL_NAME}")
print(f"   Temperature : {TEMPERATURE}")
print(f"   Alpha correct / incorrect : {ALPHA_CORRECT} / {ALPHA_INCORRECT}")

c:\Users\info\anaconda3\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


✅ LLM initialisé : llama-3.3-70b-versatile
   Temperature : 0.1
   Alpha correct / incorrect : 0.25 / 0.1


---
## Cellule 3 — AgentState : le contrat de données

Le `AgentState` est un dictionnaire typé (`TypedDict`) partagé entre **tous les nœuds** du graphe.  
Chaque nœud **lit** certains champs et **écrit** d'autres.

| Champ | Qui le remplit | Qui le lit |
|---|---|---|
| `question` | Agent générateur | retrieve, llm_infer_answer, mcq_check, gap_analysis |
| `student_answer` | Agent générateur | mcq_check, ambiguity_check |
| `sources` | Agent générateur | retrieve |
| `student_level` | Agent générateur | update_level |
| `formatted_context` | retrieve | llm_infer_answer, gap_analysis |
| `llm_inferred_answer` | **llm_infer_answer** *(nouveau)* | mcq_check |
| `correction_result` | mcq_check | gap_analysis, update_level |
| `identified_gaps` | gap_analysis / hitl | Agent feedback |
| `updated_level` | update_level | Agent générateur |
| `correction_reasoning` | gap_analysis | Agent feedback |


In [3]:
from typing import TypedDict, List, Optional, Literal


class MCQQuestion(TypedDict):
    """
    Représente une question QCM complète.
    Fournie par l'agent générateur avant l'entrée dans la correction.
    """
    question_id   : str
    question_text : str
    choices       : dict   # {"A": "...", "B": "...", "C": "...", "D": "..."}
    correct_answer: str    # ex: "B"
    question_type : Literal["mcq", "open"]


class AgentState(TypedDict):
    """
    State partagé entre tous les nœuds.
    LangGraph le passe de nœud en nœud, chacun peut le modifier.
    """
    # ── Entrées (remplies par l'agent générateur) ─────────────────────────
    question        : MCQQuestion
    student_answer  : str
    sources         : List[dict]   # [{"content": "...", "metadata": {...}}]
    student_level   : float        # Niveau entre 0.0 et 1.0
    history         : List[dict]   # Historique des exercices passés

    # ── Champ intermédiaire (usage interne) ───────────────────────────────
    formatted_context    : str        # Sources formatées pour le prompt LLM
    llm_inferred_answer  : str        # Bonne réponse inférée par le LLM depuis les sources

    # ── Sorties produites par l'agent de correction ───────────────────────
    correction_result   : dict         # Score, verdict, analyse
    identified_gaps     : List[str]    # Concepts non maîtrisés
    updated_level       : float        # Niveau recalculé
    correction_reasoning: str          # Raisonnement pour l'agent feedback

    # ── HITL ──────────────────────────────────────────────────────────────
    needs_human_review : bool
    human_feedback     : Optional[str]


print("✅ AgentState défini avec", len(AgentState.__annotations__), "champs")

✅ AgentState défini avec 13 champs


---
## Cellule 4 — Nœud 1 : `retrieve_relevant_chunks`

**Rôle** : Préparer le contexte RAG avant toute correction.  
**Entrée** : `state["sources"]` — liste de dicts `{content, metadata}`  
**Sortie** : `state["formatted_context"]` — string structuré injecté dans le prompt LLM

> Ce nœud ne prend aucune décision. Il formate uniquement.  
> En production, il pourrait aussi re-requêter le vector store (ChromaDB / FAISS).

In [4]:
def retrieve_relevant_chunks(state: AgentState) -> AgentState:
    """
    Nœud 1 — Formatage des sources RAG.

    Entrée  : state["sources"]           → liste de chunks du cours
    Sortie  : state["formatted_context"] → texte structuré pour le LLM
    """
    sources = state.get("sources", [])

    if not sources:
        state["formatted_context"] = "Aucune source disponible."
        print("  ⚠️  Aucune source RAG fournie")
        return state

    context_blocks = []
    for i, src in enumerate(sources):
        content  = src.get("content", "")
        metadata = src.get("metadata", {})
        chapter  = metadata.get("chapter", "N/A")
        page     = metadata.get("page", "N/A")
        context_blocks.append(
            f"[Source {i+1} — Chapitre {chapter}, page {page}]\n{content}"
        )

    state["formatted_context"] = "\n\n".join(context_blocks)

    print(f"  📚 {len(sources)} source(s) formatée(s) → formatted_context prêt")
    return state


print("✅ Nœud 1 'retrieve_relevant_chunks' défini")

✅ Nœud 1 'retrieve_relevant_chunks' défini


---
## Cellule 4b — Nœud 1b : `llm_infer_answer`

**Rôle** : Demander au LLM de **deviner la bonne réponse** en se basant uniquement sur les sources RAG.  
**Entrée** : `state["formatted_context"]` + `state["question"]`  
**Sortie** : `state["llm_inferred_answer"]` — lettre A/B/C/D inférée par le modèle

> Ce nœud s'insère entre `retrieve` et `mcq_check`.  
> Le champ `correct_answer` dans la question **n'est plus utilisé** par `mcq_check` — c'est `llm_inferred_answer` qui fait foi.


In [5]:
from langchain_core.prompts import ChatPromptTemplate

MCQ_INFER_ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Tu es un expert pédagogique.
On te donne une question QCM et des extraits de cours.
Ton rôle : lire les sources et déterminer quelle lettre (A, B, C ou D) est la bonne réponse.

Réponds UNIQUEMENT avec la lettre de la bonne réponse (A, B, C ou D), rien d'autre.
Ne justifie pas. Ne commente pas. Juste la lettre."""),

    ("human", """
## Question
{question_text}

## Choix proposés
{choices_formatted}

## Sources du cours
{formatted_context}

Quelle lettre est la bonne réponse ?
""")
])


def llm_infer_answer(state: AgentState) -> AgentState:
    """
    Nœud 1b — Inférence de la bonne réponse par le LLM.

    Entrée  : state["formatted_context"]  → sources RAG formatées
              state["question"]           → question + choix
    Sortie  : state["llm_inferred_answer"] → lettre A/B/C/D devinée par le LLM

    Ce nœud remplace l'usage de question["correct_answer"] dans mcq_check.
    """
    question = state["question"]
    choices_formatted = "\n".join([
        f"  {k}) {v}" for k, v in question["choices"].items()
    ])

    print("  🤖 LLM infère la bonne réponse depuis les sources...")

    chain    = MCQ_INFER_ANSWER_PROMPT | llm
    response = chain.invoke({
        "question_text"    : question["question_text"],
        "choices_formatted": choices_formatted,
        "formatted_context": state["formatted_context"]
    })

    inferred = response.content.strip().upper()
    # Sécurité : garder uniquement la première lettre valide
    inferred = next((c for c in inferred if c in question["choices"]), "A")

    state["llm_inferred_answer"] = inferred
    print(f"  🎯 Réponse inférée par le LLM : {inferred} — '{question['choices'].get(inferred, '?')}'")
    return state


print("✅ Nœud 1b 'llm_infer_answer' défini")


✅ Nœud 1b 'llm_infer_answer' défini


---
## Cellule 5 — Nœud 2 : `deterministic_mcq_check`

**Rôle** : Correction binaire et certaine — **aucun appel LLM**.  
**Entrée** : `state["student_answer"]` + `state["llm_inferred_answer"]` *(inféré par le LLM)*  
**Sortie** : `state["correction_result"]` avec score, verdict, textes des choix

> ⚠️ Ce nœud utilise maintenant `llm_inferred_answer` comme référence au lieu de `question["correct_answer"]`.

Ce nœud produit :
- `score` : `1.0` si correct, `0.0` si incorrect
- `verdict` : `"correct"` ou `"incorrect"`
- `confidence` : toujours `1.0` pour un QCM
- `valid_choice` : `False` si l'étudiant a tapé autre chose que A/B/C/D → déclenche le HITL


In [6]:
def deterministic_mcq_check(state: AgentState) -> AgentState:
    """
    Nœud 2 — Correction déterministe sans LLM.

    Entrée  : state["student_answer"]          → réponse choisie ex: "A"
              state["question"]["correct_answer"] → clé correcte ex: "B"
    Sortie  : state["correction_result"]        → dict complet
    """
    question       = state["question"]
    student_answer = state["student_answer"].strip().upper()
    # Utilise la réponse inférée par le LLM au lieu de question["correct_answer"]
    correct_answer = state.get("llm_inferred_answer", question.get("correct_answer", "")).strip().upper()
    choices        = question["choices"]

    is_correct   = student_answer == correct_answer
    score        = 1.0 if is_correct else 0.0
    valid_choice = student_answer in choices

    # Récupération des textes associés aux choix
    chosen_text  = choices.get(student_answer, "Réponse non reconnue")
    correct_text = choices.get(correct_answer, "")

    state["correction_result"] = {
        "score"          : score,
        "verdict"        : "correct" if is_correct else "incorrect",
        "student_answer" : student_answer,
        "correct_answer" : correct_answer,
        "chosen_text"    : chosen_text,
        "correct_text"   : correct_text,
        "is_correct"     : is_correct,
        "confidence"     : 1.0,      # Toujours certain pour un QCM
        "ambiguous"      : False,
        "valid_choice"   : valid_choice
    }

    status = "✅ CORRECT" if is_correct else "❌ INCORRECT"
    print(f"  {status} | Étudiant: {student_answer} → '{chosen_text}'")
    print(f"           | Bonne réponse: {correct_answer} → '{correct_text}'")
    print(f"           | Score: {score} | Choix valide: {valid_choice}")
    return state


print("✅ Nœud 2 'deterministic_mcq_check' défini")

✅ Nœud 2 'deterministic_mcq_check' défini


---
## Cellule 6 — Nœud 3 : `ambiguity_check` (routeur)

**Rôle** : Décider vers quel nœud continuer. **Ne modifie pas le state.**  
**Entrée** : `state["correction_result"]["valid_choice"]`  
**Sortie** : string `"hitl"` ou `"gap_analysis"` — consommé par LangGraph

```
valid_choice == False  →  "hitl"         (réponse hors A/B/C/D)
valid_choice == True   →  "gap_analysis" (réponse normale)
```

In [7]:
def ambiguity_check(state: AgentState) -> str:
    """
    Nœud 3 — Routeur conditionnel.

    Entrée  : state["correction_result"]["valid_choice"]
    Sortie  : "hitl" ou "gap_analysis" (string consommé par LangGraph)

    Ce nœud ne modifie PAS le state.
    Pour un QCM, le seul cas HITL est une réponse hors des choix valides.
    """
    valid_choice = state["correction_result"].get("valid_choice", False)

    if not valid_choice:
        state["needs_human_review"] = True
        print("  🔀 Routage → HITL  (réponse invalide)")
        return "hitl"

    state["needs_human_review"] = False
    print("  🔀 Routage → gap_analysis  (réponse valide)")
    return "gap_analysis"


print("✅ Nœud 3 'ambiguity_check' défini")

✅ Nœud 3 'ambiguity_check' défini


---
## Cellule 7 — Nœud 4a : `hitl_node` (Human-in-the-Loop)

**Rôle** : Suspendre l'exécution du graphe pour intervention humaine.  
**Entrée** : state complet avec réponse invalide  
**Sortie** : `state["human_feedback"]` + correction du score si fournie

**Mécanisme LangGraph** :
1. `interrupt()` sérialise le state dans le **checkpointer** (MemorySaver)
2. L'exécution est **suspendue** jusqu'à reprise manuelle
3. Le superviseur fournit `{"score": 0.5, "gaps": [...], "comment": "..."}`
4. LangGraph reprend depuis ce nœud avec le state mis à jour

> ⚠️ Ce nœud n'est atteint que si `valid_choice == False`

In [8]:
from langgraph.types import interrupt


def hitl_node(state: AgentState) -> AgentState:
    """
    Nœud 4a — Human-in-the-Loop.

    Entrée  : state complet avec réponse invalide
    Sortie  : state["human_feedback"] + correction éventuelle du score

    LangGraph suspend le graphe via interrupt() et attend la reprise.
    Le superviseur fournit un dict :
      {
        "score"  : 0.0 à 1.0,
        "gaps"   : ["concept 1", ...],
        "comment": "explication libre"
      }
    """
    print("  ⏸️  HITL déclenché — suspension du graphe")
    print(f"     Question : {state['question']['question_text'][:60]}...")
    print(f"     Réponse invalide : '{state['student_answer']}'")

    # Suspension — le superviseur voit cette payload dans son interface
    human_input = interrupt({
        "message"        : "Réponse invalide — révision humaine requise",
        "question_text"  : state["question"]["question_text"],
        "choices"        : state["question"]["choices"],
        "student_answer" : state["student_answer"],
        "sources_count"  : len(state["sources"])
    })

    # Intégration de la décision humaine dans le state
    if human_input:
        state["human_feedback"] = human_input.get("comment", "")

        if "score" in human_input:
            score = float(human_input["score"])
            state["correction_result"]["score"]      = score
            state["correction_result"]["is_correct"] = score >= 0.5
            state["correction_result"]["verdict"]    = (
                "correct" if score >= 0.5 else "incorrect"
            )
            print(f"  👤 Score humain : {score}")

        if "gaps" in human_input:
            state["identified_gaps"] = human_input["gaps"]
            print(f"  👤 Gaps humains : {human_input['gaps']}")

    print("  ▶️  Reprise après intervention humaine")
    return state


print("✅ Nœud 4a 'hitl_node' défini")

✅ Nœud 4a 'hitl_node' défini


---
## Cellule 8 — Nœud 4b : `llm_gap_analysis`

**Rôle** : Analyser **pourquoi** l'étudiant s'est trompé en se basant sur les sources RAG.  
**Entrée** : `correction_result`, `question`, `formatted_context`  
**Sortie** : `identified_gaps`, `correction_reasoning`, enrichissement de `correction_result`

**Logique** :
- Si `is_correct == True` → **pas d'appel LLM**, sortie rapide
- Si `is_correct == False` → appel **Groq** avec le prompt d'analyse

Le LLM répond en JSON structuré :
```json
{
  "likely_misconception" : "L'étudiant confond itération et récursion",
  "gaps"                 : ["définition récursivité", "différence boucle/récursion"],
  "distractor_analysis"  : "Le choix A évoque la répétition...",
  "source_references"    : ["La récursivité diffère de l'itération car..."]
}
```

In [9]:
import json
from langchain_core.prompts import ChatPromptTemplate


MCQ_GAP_ANALYSIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Tu es un analyste pédagogique expert.
Un étudiant vient de répondre incorrectement à un QCM.
Ton rôle : identifier POURQUOI il s'est trompé
en te basant STRICTEMENT sur les sources du cours fournies.

Réponds UNIQUEMENT en JSON valide, sans texte autour, sans balises markdown :
{{
  "likely_misconception" : "<idée fausse probable de l'étudiant>",
  "gaps"                 : ["<concept non maîtrisé 1>", "<concept 2>"],
  "distractor_analysis"  : "<pourquoi ce choix erroné semblait plausible>",
  "source_references"    : ["<extrait court justifiant la bonne réponse>"]
}}"""),

    ("human", """
## Question
{question_text}

## Choix proposés
{choices_formatted}

## Réponse de l'étudiant : {student_answer} — "{chosen_text}"
## Bonne réponse       : {correct_answer} — "{correct_text}"

## Sources du cours
{formatted_context}

Analyse l'erreur de l'étudiant en te basant sur les sources.
""")
])


def llm_gap_analysis(state: AgentState) -> AgentState:
    """
    Nœud 4b — Analyse LLM des lacunes (uniquement si réponse incorrecte).

    Entrée  : state["correction_result"]["is_correct"]
              state["question"], state["formatted_context"]
    Sortie  : state["identified_gaps"]       → liste de concepts
              state["correction_reasoning"]  → JSON brut pour agent feedback
    """
    result = state["correction_result"]

    # ── Cas réponse correcte : pas d'analyse nécessaire ──────────────────
    if result["is_correct"]:
        state["identified_gaps"]      = []
        state["correction_reasoning"] = "Réponse correcte."
        print("  ⚡ Réponse correcte → pas d'appel LLM (sortie rapide)")
        return state

    # ── Cas réponse incorrecte : analyse LLM ──────────────────────────────
    print("  🤖 Appel Groq pour analyse des lacunes...")

    question = state["question"]
    choices_formatted = "\n".join([
        f"  {k}) {v}" for k, v in question["choices"].items()
    ])

    chain    = MCQ_GAP_ANALYSIS_PROMPT | llm
    response = chain.invoke({
        "question_text"    : question["question_text"],
        "choices_formatted": choices_formatted,
        "student_answer"   : result["student_answer"],
        "chosen_text"      : result["chosen_text"],
        "correct_answer"   : result["correct_answer"],
        "correct_text"     : result["correct_text"],
        "formatted_context": state["formatted_context"]
    })

    # ── Nettoyage et parsing JSON ─────────────────────────────────────────
    raw = response.content.strip()
    if raw.startswith("```"):
        parts = raw.split("```")
        raw   = parts[1] if len(parts) > 1 else raw
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()

    analysis = json.loads(raw)

    # ── Enrichissement du state ───────────────────────────────────────────
    state["correction_result"].update(analysis)
    state["identified_gaps"]      = analysis.get("gaps", [])
    state["correction_reasoning"] = json.dumps(analysis, ensure_ascii=False)

    print(f"  📋 Lacunes identifiées : {state['identified_gaps']}")
    print(f"  💡 Idée fausse : {analysis.get('likely_misconception', '')}")
    return state


print("✅ Nœud 4b 'llm_gap_analysis' défini")

✅ Nœud 4b 'llm_gap_analysis' défini


---
## Cellule 9 — Nœud 5 : `update_level_node`

**Rôle** : Recalculer le niveau de l'étudiant après la correction.  
**Entrée** : `state["correction_result"]["score"]` + `state["student_level"]`  
**Sortie** : `state["updated_level"]`

**Formule de mise à jour** :
```
nouveau_niveau = α × score + (1 − α) × niveau_précédent
```

**Paramètres asymétriques** :
- `α = 0.25` si bonne réponse → progression douce
- `α = 0.10` si mauvaise réponse → régression encore plus douce

Cette asymétrie évite qu'un étudiant perde son niveau accumulé sur une seule erreur.

In [10]:
def update_level_node(state: AgentState) -> AgentState:
    """
    Nœud 5 — Mise à jour du niveau étudiant.

    Entrée  : state["correction_result"]["score"]
              state["student_level"]
    Sortie  : state["updated_level"]  → lu par l'agent générateur

    Formule : nouveau = α × score + (1−α) × ancien
    α = ALPHA_CORRECT si score=1.0, sinon ALPHA_INCORRECT
    Résultat borné entre LEVEL_MIN et LEVEL_MAX.
    """
    score          = state["correction_result"]["score"]
    previous_level = state["student_level"]

    # Coefficient asymétrique
    alpha     = ALPHA_CORRECT if score == 1.0 else ALPHA_INCORRECT
    new_level = alpha * score + (1 - alpha) * previous_level
    new_level = round(max(LEVEL_MIN, min(LEVEL_MAX, new_level)), 3)

    state["updated_level"] = new_level

    direction = "📈" if new_level > previous_level else "📉"
    print(f"  {direction} Niveau : {previous_level:.3f} → {new_level:.3f}  (α={alpha})")
    return state


print("✅ Nœud 5 'update_level_node' défini")

✅ Nœud 5 'update_level_node' défini


---
## Cellule 10 — Assemblage du graphe LangGraph

On connecte les 6 nœuds avec leurs edges et l'edge conditionnel.

```
retrieve → infer_answer → mcq_check → (ambiguity_check) → hitl ──┐
                                            ↓                    │
                                      gap_analysis  ◄────────────┘
                                            ↓
                                      update_level
                                            ↓
                                          END
```

**Points clés** :
- `infer_answer` : **nouveau nœud** — le LLM lit les sources et devine la bonne réponse
- `mcq_check` compare maintenant `student_answer` avec `llm_inferred_answer`
- `MemorySaver` : checkpointer qui persiste le state entre les étapes (nécessaire pour HITL)
- `interrupt_before=["hitl"]` : LangGraph pause automatiquement avant ce nœud
- `thread_id` : identifiant unique par étudiant/session pour isoler les states


In [11]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


def build_correction_agent():
    """Construit et compile le graphe LangGraph de l'agent de correction."""

    graph = StateGraph(AgentState)

    # ── Enregistrement des nœuds ─────────────────────────────────────────
    graph.add_node("retrieve"      , retrieve_relevant_chunks)
    graph.add_node("infer_answer"  , llm_infer_answer)          # ← NOUVEAU
    graph.add_node("mcq_check"     , deterministic_mcq_check)
    graph.add_node("hitl"        , hitl_node)
    graph.add_node("gap_analysis", llm_gap_analysis)
    graph.add_node("update_level", update_level_node)

    # ── Edges séquentiels ────────────────────────────────────────────────
    graph.set_entry_point("retrieve")
    graph.add_edge("retrieve"     , "infer_answer")  # retrieve → infer LLM
    graph.add_edge("infer_answer" , "mcq_check")    # infer → correction

    # ── Edge conditionnel après mcq_check ────────────────────────────────
    graph.add_conditional_edges(
        "mcq_check",
        ambiguity_check,           # Fonction de routage
        {
            "hitl"        : "hitl",
            "gap_analysis": "gap_analysis"
        }
    )

    # ── Après HITL → reprend sur gap_analysis ────────────────────────────
    graph.add_edge("hitl"        , "gap_analysis")
    graph.add_edge("gap_analysis", "update_level")
    graph.add_edge("update_level", END)

    # ── Checkpointer : persistance state pour HITL ───────────────────────
    checkpointer = MemorySaver()

    return graph.compile(
        checkpointer=checkpointer,
        interrupt_before=["hitl"]   # Pause automatique avant HITL
    )


# Compilation
correction_agent = build_correction_agent()
print("✅ Graphe LangGraph compilé avec succès")
print("   Nœuds :", ["retrieve", "mcq_check", "ambiguity_check", "hitl", "gap_analysis", "update_level"])
print("   Checkpointer : MemorySaver (in-memory)")
print("   HITL : interrupt_before=['hitl']")

✅ Graphe LangGraph compilé avec succès
   Nœuds : ['retrieve', 'mcq_check', 'ambiguity_check', 'hitl', 'gap_analysis', 'update_level']
   Checkpointer : MemorySaver (in-memory)
   HITL : interrupt_before=['hitl']


---
## Cellule 11 — Test cas 1 : Réponse incorrecte

L'étudiant répond **"A"** alors que la bonne réponse est **"B"**.  
On s'attend à :
- `score = 0.0`, `verdict = "incorrect"`
- Appel LLM pour analyser l'erreur
- `identified_gaps` non vide
- Niveau qui baisse légèrement (α = 0.10)

In [17]:
# ── State initial — fourni par l'agent générateur en production ───────────
input_state_incorrect = {
    "question": {
        "question_id"   : "q_001",
        "question_text" : "Qu'est-ce que la récursivité en programmation ?",
        "choices": {
            "A": "Une boucle for qui répète des instructions",
            "D": "Une fonction qui s'appelle elle-même",
            "C": "Un tableau qui contient d'autres tableaux",
            "B": "Une variable qui change de type dynamiquement"
        },
        "correct_answer": "",  # (non utilisé — le LLM infère depuis les sources) 
        "question_type" : "mcq"
    },
    "student_answer" : "A",   # ← mauvaise réponse
    "sources": [
        {
            "content" : (
                "La récursivité est un mécanisme par lequel une fonction fait appel "
                "à elle-même pour résoudre un problème. Elle diffère de l'itération "
                "car elle ne nécessite pas de boucle explicite. "
                "Chaque appel récursif traite un sous-problème plus petit "
                "jusqu'à atteindre un cas de base."
            ),
            "metadata": {"chapter": "5", "page": "112"}
        },
        {
            "content" : (
                "Une boucle for est une structure itérative. "
                "Elle répète un bloc d'instructions un nombre déterminé de fois "
                "sans que la fonction ne s'appelle elle-même. "
                "L'itération et la récursivité produisent des résultats similaires "
                "mais avec des mécanismes fondamentalement différents."
            ),
            "metadata": {"chapter": "3", "page": "67"}
        }
    ],
    "student_level"     : 0.45,
    "history"           : [],
    "needs_human_review": False,
    "human_feedback"    : None
}

# ── Exécution ──────────────────────────────────────────────────────────────
print("=" * 60)
print("TEST CAS 1 — Réponse incorrecte")
print("=" * 60)

config_1 = {"configurable": {"thread_id": "student_42_session_1"}}
result_1 = correction_agent.invoke(input_state_incorrect, config=config_1)

# ── Affichage des résultats ────────────────────────────────────────────────
print("\n" + "─" * 40)
print("RÉSULTAT FINAL")
print("─" * 40)
correction = result_1["correction_result"]
print(f"Score   : {correction['score']}")
print(f"Verdict : {correction['verdict'].upper()}")

gaps = result_1.get("identified_gaps", [])
if gaps:
    print(f"\nLacunes identifiées :")
    for g in gaps:
        print(f"  • {g}")

if result_1.get("correction_reasoning") != "Réponse correcte.":
    reasoning = json.loads(result_1["correction_reasoning"])
    print(f"\nIdée fausse probable  : {reasoning.get('likely_misconception', '')}")
    print(f"Analyse du distracteur: {reasoning.get('distractor_analysis', '')}")

print(f"\nNiveau : {input_state_incorrect['student_level']} → {result_1['updated_level']}")
print("\n→ Transmis à l'agent feedback : correction_result, identified_gaps, correction_reasoning")
print("→ Transmis à l'agent générateur : updated_level =", result_1["updated_level"])

TEST CAS 1 — Réponse incorrecte
  📚 2 source(s) formatée(s) → formatted_context prêt
  🤖 LLM infère la bonne réponse depuis les sources...
  🎯 Réponse inférée par le LLM : D — 'Une fonction qui s'appelle elle-même'
  ❌ INCORRECT | Étudiant: A → 'Une boucle for qui répète des instructions'
           | Bonne réponse: D → 'Une fonction qui s'appelle elle-même'
           | Score: 0.0 | Choix valide: True
  🔀 Routage → gap_analysis  (réponse valide)
  🤖 Appel Groq pour analyse des lacunes...
  📋 Lacunes identifiées : ['Compréhension de la récursivité', 'Différence entre itération et récursivité']
  💡 Idée fausse : L'étudiant confond récursivité et itération
  📉 Niveau : 0.450 → 0.405  (α=0.1)

────────────────────────────────────────
RÉSULTAT FINAL
────────────────────────────────────────
Score   : 0.0
Verdict : INCORRECT

Lacunes identifiées :
  • Compréhension de la récursivité
  • Différence entre itération et récursivité

Idée fausse probable  : L'étudiant confond récursivité et itéra

---
## Cellule 12 — Test cas 2 : Réponse correcte

L'étudiant répond **"B"** — la bonne réponse.  
On s'attend à :
- `score = 1.0`, `verdict = "correct"`
- **Pas d'appel LLM** → sortie rapide du nœud `gap_analysis`
- `identified_gaps = []`
- Niveau qui monte (α = 0.25)

In [13]:

input_state_correct = {
    **input_state_incorrect,          # Même question, même sources
    "student_answer" : "b",           # ← bonne réponse
    "student_level"  : 0.45
}

print("=" * 60)
print("TEST CAS 2 — Réponse correcte")
print("=" * 60)

config_2 = {"configurable": {"thread_id": "student_42_session_2"}}
result_2 = correction_agent.invoke(input_state_correct, config=config_2)

print("\n" + "─" * 40)
print("RÉSULTAT FINAL")
print("─" * 40)
correction_2 = result_2["correction_result"]
print(f"Score   : {correction_2['score']}")
print(f"Verdict : {correction_2['verdict'].upper()}")
print(f"Lacunes : {result_2.get('identified_gaps', [])}  (vide = correct)")
print(f"\nNiveau : {input_state_correct['student_level']} → {result_2['updated_level']}")
print("\n→ Aucun appel LLM effectué (sortie rapide)")

TEST CAS 2 — Réponse correcte
  📚 2 source(s) formatée(s) → formatted_context prêt
  🤖 LLM infère la bonne réponse depuis les sources...
  🎯 Réponse inférée par le LLM : B — 'Une fonction qui s'appelle elle-même'
  ✅ CORRECT | Étudiant: B → 'Une fonction qui s'appelle elle-même'
           | Bonne réponse: B → 'Une fonction qui s'appelle elle-même'
           | Score: 1.0 | Choix valide: True
  🔀 Routage → gap_analysis  (réponse valide)
  ⚡ Réponse correcte → pas d'appel LLM (sortie rapide)
  📈 Niveau : 0.450 → 0.588  (α=0.25)

────────────────────────────────────────
RÉSULTAT FINAL
────────────────────────────────────────
Score   : 1.0
Verdict : CORRECT
Lacunes : []  (vide = correct)

Niveau : 0.45 → 0.588

→ Aucun appel LLM effectué (sortie rapide)


---
## Cellule 13 — Test cas 3 : Réponse invalide (HITL)

L'étudiant tape **"je sais pas"** au lieu d'une lettre.  
On s'attend à :
- `ambiguity_check` → route vers `"hitl"`
- Graphe **suspendu** avant le nœud HITL
- Reprise manuelle avec un score fourni par le superviseur

> ⚠️ Ce test simule la reprise HITL avec `update()`.  
> En production, la reprise viendrait de l'interface superviseur.

In [14]:
from langgraph.types import Command

In [15]:
input_state_invalid = {
    **input_state_incorrect,
    "student_answer" : "je sais pas",   # ← réponse invalide
    "student_level"  : 0.45
}

print("=" * 60)
print("TEST CAS 3 — Réponse invalide → HITL")
print("=" * 60)

config_3 = {"configurable": {"thread_id": "student_42_session_3"}}

# ── Première exécution : s'arrête avant hitl_node ─────────────────────────
print("\n[Étape 1] Exécution initiale...")
result_paused = correction_agent.invoke(input_state_invalid, config=config_3)

# LangGraph renvoie None ou un state partiel selon la version
# On vérifie le prochain nœud attendu
state_snapshot = correction_agent.get_state(config_3)
print(f"\n  Prochain nœud : {state_snapshot.next}")
print("  → Graphe suspendu, en attente d'intervention humaine")

# ── Simulation de l'intervention humaine ─────────────────────────────────
print("\n[Étape 2] Intervention du superviseur humain...")
human_decision = {
    "score"  : 0.0,
    "gaps"   : ["compréhension de la consigne", "définition récursivité"],
    "comment": "L'étudiant n'a pas fourni de réponse valide — score 0 attribué"
}
print(f"  Décision humaine : {human_decision}")

# ── Reprise du graphe avec la décision humaine ────────────────────────────
print("\n[Étape 3] Reprise du graphe...")
result_resumed = correction_agent.invoke(
    Command(resume=human_decision),
    config=config_3
)

print("\n" + "─" * 40)
print("RÉSULTAT FINAL (après HITL)")
print("─" * 40)
print(f"Score   : {result_resumed['correction_result']['score']}")
print(f"Verdict : {result_resumed['correction_result']['verdict'].upper()}")
print(f"Lacunes : {result_resumed.get('identified_gaps', [])}")
print(f"Feedback humain : {result_resumed.get('human_feedback', '')}")
print(f"Niveau  : {input_state_invalid['student_level']} → {result_resumed.get('updated_level', 'N/A')}")

TEST CAS 3 — Réponse invalide → HITL

[Étape 1] Exécution initiale...
  📚 2 source(s) formatée(s) → formatted_context prêt
  🤖 LLM infère la bonne réponse depuis les sources...
  🎯 Réponse inférée par le LLM : B — 'Une fonction qui s'appelle elle-même'
  ❌ INCORRECT | Étudiant: JE SAIS PAS → 'Réponse non reconnue'
           | Bonne réponse: B → 'Une fonction qui s'appelle elle-même'
           | Score: 0.0 | Choix valide: False
  🔀 Routage → HITL  (réponse invalide)

  Prochain nœud : ('hitl',)
  → Graphe suspendu, en attente d'intervention humaine

[Étape 2] Intervention du superviseur humain...
  Décision humaine : {'score': 0.0, 'gaps': ['compréhension de la consigne', 'définition récursivité'], 'comment': "L'étudiant n'a pas fourni de réponse valide — score 0 attribué"}

[Étape 3] Reprise du graphe...
  ⏸️  HITL déclenché — suspension du graphe
     Question : Qu'est-ce que la récursivité en programmation ?...
     Réponse invalide : 'je sais pas'
  👤 Score humain : 0.0
  👤 Gaps h

---
## Récapitulatif — Ce que produit l'agent de correction

| Champ produit | Destinataire | Contenu |
|---|---|---|
| `correction_result` | Agent feedback | Score, verdict, textes des choix, analyse |
| `identified_gaps` | Agent feedback | Liste des concepts non maîtrisés |
| `correction_reasoning` | Agent feedback | JSON brut de l'analyse LLM |
| `updated_level` | Agent générateur | Nouveau niveau pour calibrer la prochaine question |

## Extension possible

- Remplacer `MemorySaver` par `SqliteSaver` pour une persistance réelle entre sessions
- Brancher un vrai vector store (ChromaDB, FAISS) dans `retrieve_relevant_chunks`
- Ajouter un nœud de logging pour tracer les erreurs fréquentes par concept
- Étendre `MCQQuestion` pour gérer les QCM à choix multiples (plusieurs bonnes réponses)